# 06 — Concentration & monitoring-pack report

**Plain English:** Where is the risk *concentrated*? We slice exposure and
default by geography (state) and by vintage, then assemble a short
**monitoring-pack report** that pulls the real numbers from every prior notebook.

**One-line terms**
- **Concentration** — how exposure clusters in a few states / cohorts; a portfolio risk in itself.
- **Exposure (UPB)** — current unpaid principal balance, i.e. money still at risk.
- **APS 330-style** — laid out like an APRA APS 330 credit-risk disclosure table. *Format only — illustrative, not a regulatory submission.*

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# Latest snapshot per loan for a point-in-time concentration view --------
latest = panel.sort_values(["loan_seq", "mob"]).groupby("loan_seq").tail(1)
active = latest[latest.zb_code.isna()].copy()  # still on book

by_state = (active.groupby("prop_state")
            .agg(loans=("loan_seq", "size"),
                 exposure_upb=("cur_upb", "sum"),
                 pct_90plus=("bucket", lambda s: round(100 * s.isin(["90+", "Default"]).mean(), 2)))
            .sort_values("exposure_upb", ascending=False))
by_state["exposure_share_pct"] = (100 * by_state.exposure_upb / by_state.exposure_upb.sum()).round(2)
by_state.head(12)

,loans,exposure_upb,pct_90plus,exposure_share_pct
prop_state,,,,
CA,1727,2.995965e+08,0.46,17.26
NY,820,1.270075e+08,0.85,7.32
FL,955,1.095006e+08,0.94,6.31
TX,1017,1.032622e+08,0.69,5.95
IL,697,7.439506e+07,1.15,4.29
VA,451,6.175894e+07,0.22,3.56
NJ,428,6.135025e+07,0.47,3.53
PA,587,5.881619e+07,0.68,3.39
MA,336,5.157980e+07,0.60,2.97


In [3]:
# Concentration by vintage (APS 330-style layout) -----------------------
by_vintage = (active.groupby("vintage")
              .agg(loans=("loan_seq", "size"),
                   exposure_upb=("cur_upb", "sum"),
                   avg_credit_score=("credit_score", "mean"),
                   pct_stage2=("stage", lambda s: round(100 * (s == "Stage 2").mean(), 2)),
                   pct_stage3=("stage", lambda s: round(100 * (s == "Stage 3").mean(), 2)))
              .reset_index())
by_vintage["exposure_upb"] = by_vintage.exposure_upb.round(0)
by_vintage["avg_credit_score"] = by_vintage.avg_credit_score.round(0)
by_vintage.to_csv(m.OUT_TABLES / "06_concentration_vintage.csv", index=False)
by_state.round(2).to_csv(m.OUT_TABLES / "06_concentration_state.csv")
by_vintage

,vintage,loans,exposure_upb,avg_credit_score,pct_stage2,pct_stage3
0,2007,1378,1.140695e+08,706.0,4.14,2.03
1,2008,1163,1.097066e+08,736.0,3.35,1.46
2,2015,11976,1.511844e+09,750.0,1.29,0.47


In [4]:
# Assemble the monitoring-pack report from real outputs -----------------
T = m.OUT_TABLES
rep = ROOT_REPORT = (m.REPO_ROOT / "outputs" / "report")
rep.mkdir(parents=True, exist_ok=True)

probs_b = pd.read_csv(T / "02_bucket_transition_matrix.csv", index_col=0)
rr = pd.read_csv(T / "02_roll_rates.csv")
moves = pd.read_csv(T / "03_stage_movements.csv")
vint = pd.read_csv(T / "05_vintage_cumulative_default.csv")
wsumm = pd.read_csv(T / "04_watchlist_summary.csv")

lines = []
lines.append("# Portfolio Monitoring Pack — loan-level (Freddie Mac SFLLD)\n")
lines.append("_Real loan-level mortgage data. The monitoring mechanics apply equally to "
             "commercial loan portfolios with a monthly status feed._\n")
lines.append("## 1. Monthly delinquency-bucket transition matrix\n")
lines.append(probs_b.round(4).to_markdown() + "\n")
lines.append("![heatmap](../charts/02_bucket_transition_heatmap.png)\n")
lines.append("## 2. Headline roll rates\n")
lines.append(rr.to_markdown(index=False) + "\n")
lines.append("## 3. IFRS 9 stage movements (loan-months)\n")
lines.append(moves.to_markdown(index=False) + "\n")
lines.append("## 4. Early-warning watchlist (by vintage / stage)\n")
lines.append(wsumm.to_markdown(index=False) + "\n")
lines.append("## 5. Vintage tracking — cumulative default by months on book\n")
lines.append(vint.to_markdown(index=False) + "\n")
lines.append("![vintage curves](../charts/05_vintage_default_curves.png)\n")
lines.append("## 6. Concentration by state (top 10) — APS 330-style format\n")
lines.append("_Format only — illustrative, not a regulatory submission._\n")
lines.append(by_state.head(10).round(2).to_markdown() + "\n")

(rep / "monitoring_pack.md").write_text("\n".join(lines), encoding="utf-8")
print("wrote ->", rep / "monitoring_pack.md")

wrote -> D:\Jane\Job Search\Github\bank\github project\mortgage-portfolio-monitoring\outputs\report\monitoring_pack.md
